In [ ]:
import sys
import subprocess

required = ["sentence-transformers", "datasets", "scipy", "pandas", "scikit-learn"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *required])


In [ ]:
import random
import time
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics.pairwise import paired_cosine_distances

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "sentence-transformers/all-MiniLM-L6-v2"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 128 if device == "mps" else 64
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "batch_size": batch_size,
    "seed": seed,
})


In [ ]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
print({"num_examples": len(df), "columns": df.columns.tolist()})
print(df.head())


In [ ]:
model = SentenceTransformer(model_name, device=device)
model.eval()
print(model_name)


In [ ]:
sentences1 = df["sentence1"].tolist()
sentences2 = df["sentence2"].tolist()
labels = df["label"].to_numpy(dtype=np.float32)

emb1 = model.encode(
    sentences1,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=False,
    show_progress_bar=False,
)

emb2 = model.encode(
    sentences2,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=False,
    show_progress_bar=False,
)

embedding_dim = int(emb1.shape[1])
cosine_similarity_raw = 1.0 - paired_cosine_distances(emb1, emb2)
predicted_score_0_5 = 2.5 * (cosine_similarity_raw + 1.0)

print({
    "embedding_dim": embedding_dim,
    "emb1_shape": emb1.shape,
    "emb2_shape": emb2.shape,
})


In [ ]:
pearson_raw = pearsonr(cosine_similarity_raw, labels).statistic
spearman_raw = spearmanr(cosine_similarity_raw, labels).statistic
pearson_rescaled = pearsonr(predicted_score_0_5, labels).statistic
spearman_rescaled = spearmanr(predicted_score_0_5, labels).statistic

results_df = df.copy()
results_df["cosine_similarity_raw"] = cosine_similarity_raw
results_df["predicted_score_0_5"] = predicted_score_0_5

print(results_df[["sentence1", "sentence2", "label", "cosine_similarity_raw", "predicted_score_0_5"]].head(10))


In [ ]:
runtime_seconds = time.time() - start_time

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"num_examples: {len(df)}")
print(f"embedding_dimensionality: {embedding_dim}")
print(f"pearson_raw_cosine: {pearson_raw:.6f}")
print(f"spearman_raw_cosine: {spearman_raw:.6f}")
print(f"pearson_rescaled_0_5: {pearson_rescaled:.6f}")
print(f"spearman_rescaled_0_5: {spearman_rescaled:.6f}")
print(f"runtime_seconds: {runtime_seconds:.2f}")
